# West Africa Urban Heat, Land Use Change and Green–Blue Cooling Analysis

Complete workflow for:
- LULCC analysis
- Landsat LST
- SUHI
- cooling-land retention
- thermal amplification
- InVEST Urban Cooling analysis
- IBM/TRM diagnostics
- Mann–Kendall trends
- XGBoost + SHAP + ALE validation

Replace input paths with your native raster/vector data.

In [ ]:
# Install packages if required
# !pip install geopandas rasterio rioxarray xgboost shap pyALE pymannkendall libpysal esda

import os
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import rioxarray as rxr
import matplotlib.pyplot as plt

from sklearn.metrics import confusion_matrix, accuracy_score, cohen_kappa_score
from sklearn.model_selection import GroupKFold
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
import shap
import pymannkendall as mk

In [ ]:
# Project folders
PROJECT = "West_Africa_UHI"
DATA = "./data"
OUTPUT = "./outputs"

os.makedirs(OUTPUT, exist_ok=True)

cities = {
    "Ouagadougou": {"area_km2":600.00},
    "Abuja_FCT": {"area_km2":7315.00},
    "AMA": {"area_km2":139.67}
}

## Landsat preprocessing

In [ ]:
def load_raster(path):
    return rxr.open_rasterio(path, masked=True)

def ndvi(red, nir):
    return (nir-red)/(nir+red)

def ndbi(swir, nir):
    return (swir-nir)/(swir+nir)

def ndwi(green, nir):
    return (green-nir)/(green+nir)

# Example:
# red = load_raster(...)
# nir = load_raster(...)
# NDVI = ndvi(red,nir)

## LULCC metrics

In [ ]:
def area_share(class_area, total_area):
    return class_area/total_area*100

def annual_sealing_pressure(built1995, built2025):
    return (built2025-built1995)/30

def cooling_land_retention(farm,forest,water_1995,farm25,forest25,water25):
    return ((farm25+forest25+water25)/(farm+forest+water_1995))*100

# Example calculations should use classified raster areas

## Land Surface Temperature and SUHI

In [ ]:
def calculate_suhi(urban_lst, reference_lst):
    return urban_lst-reference_lst

def thermal_amplification(delta_temp, delta_built_share):
    return delta_temp/(delta_built_share/10)

## InVEST Urban Cooling style calculations

In [ ]:
def cooling_capacity(shade, albedo, eti):
    return 0.6*shade + 0.2*albedo + 0.2*eti

def green_blue_synergy(r2_vw, r2_v):
    return r2_vw-r2_v

def cooling_distance_decay(hmi100,hmi300):
    return (1-hmi300/hmi100)*100

## IBM/TRM diagnostics

In [ ]:
def net_radiation(sw_down, albedo, emissivity, lw_down, sigma, Ts):
    return (1-albedo)*sw_down + emissivity*lw_down - emissivity*sigma*(Ts**4)

def bowen_ratio(H, LE):
    return H/LE

# Full IBM/TRM reconstruction requires validated atmospheric background state.

## Mann-Kendall trend analysis

In [ ]:
def mann_kendall_test(series):
    return mk.original_test(series)

# Example:
# mann_kendall_test(beta_series)

## Spatial-block XGBoost + SHAP + ALE workflow

In [ ]:
# Input dataframe should contain:
# predictors + response LST/SUHI + spatial_group_id

def train_xgb_spatial(df, features, target, groups):

    model = xgb.XGBRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=5,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    )

    gkf = GroupKFold(n_splits=5)

    results=[]

    for train,test in gkf.split(df, df[target], groups):
        model.fit(df.iloc[train][features], df.iloc[train][target])
        pred=model.predict(df.iloc[test][features])

        results.append({
            "R2":model.score(df.iloc[test][features],df.iloc[test][target])
        })

    return pd.DataFrame(results), model


# SHAP after final validated model:
# explainer = shap.TreeExplainer(model)
# shap_values = explainer(df[features])

## Export tables and figures

In [ ]:
# Save outputs
# dataframe.to_csv('./outputs/table.csv')

plt.figure(figsize=(8,5))
plt.title("Replace with manuscript figure")
plt.show()